In [37]:
import os

def count_images(folder_path):
    if os.path.exists(folder_path):
        return len([f for f in os.listdir(folder_path) 
                    if os.path.isfile(os.path.join(folder_path, f))])
    else:
        return 0

folders = [
    "/Users/indraneelpothuri/Downloads/Gemini_Images/train/ai",
    "/Users/indraneelpothuri/Downloads/Gemini_Images/train/real",
    "/Users/indraneelpothuri/Downloads/Gemini_Images/val/ai",
    "/Users/indraneelpothuri/Downloads/Gemini_Images/val/real",
    "/Users/indraneelpothuri/Downloads/Gemini_Images/test/ai",
    "/Users/indraneelpothuri/Downloads/Gemini_Images/test/real"
]

for folder in folders:
    count = count_images(folder)
    print(f"{folder}: {count}")


/Users/indraneelpothuri/Downloads/Gemini_Images/train/ai: 2216
/Users/indraneelpothuri/Downloads/Gemini_Images/train/real: 2174
/Users/indraneelpothuri/Downloads/Gemini_Images/val/ai: 646
/Users/indraneelpothuri/Downloads/Gemini_Images/val/real: 652
/Users/indraneelpothuri/Downloads/Gemini_Images/test/ai: 745
/Users/indraneelpothuri/Downloads/Gemini_Images/test/real: 683


In [38]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models, transforms, datasets
from torch.utils.data import DataLoader
import numpy as np


In [39]:
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print("Using device:", device)


Using device: mps


In [40]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])


In [41]:
base_path = "/Users/indraneelpothuri/Downloads/Gemini_Images"

train_dataset = datasets.ImageFolder(f"{base_path}/train", transform=train_transform)
val_dataset = datasets.ImageFolder(f"{base_path}/val", transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

print("Train size:", len(train_dataset))
print("Val size:", len(val_dataset))
print("Classes:", train_dataset.classes)


Train size: 4388
Val size: 1297
Classes: ['ai', 'real']


In [42]:
model = models.mobilenet_v2(pretrained=True)

# Freeze backbone
for param in model.features.parameters():
    param.requires_grad = False

# Replace classifier
model.classifier[1] = nn.Linear(model.last_channel, 2)

model = model.to(device)

print("Model ready!")


Model ready!


/Users/indraneelpothuri/miniforge3/envs/aiml/lib/python3.11/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/Users/indraneelpothuri/miniforge3/envs/aiml/lib/python3.11/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=MobileNet_V2_Weights.IMAGENET1K_V1`. You can also use `weights=MobileNet_V2_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [43]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.classifier.parameters(), lr=1e-4)


In [44]:
num_epochs = 20

for epoch in range(num_epochs):
    model.train()
    train_correct = 0
    train_total = 0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        _, predicted = torch.max(outputs, 1)
        train_total += labels.size(0)
        train_correct += (predicted == labels).sum().item()

    train_acc = train_correct / train_total

    # Validation
    model.eval()
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            _, predicted = torch.max(outputs, 1)

            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()

    val_acc = val_correct / val_total

    print(f"Epoch [{epoch+1}/{num_epochs}] "
          f"Train Acc: {train_acc:.4f} "
          f"Val Acc: {val_acc:.4f}")


Epoch [1/20] Train Acc: 0.8371 Val Acc: 0.9499
Epoch [2/20] Train Acc: 0.9505 Val Acc: 0.9661
Epoch [3/20] Train Acc: 0.9521 Val Acc: 0.9692
Epoch [4/20] Train Acc: 0.9583 Val Acc: 0.9730
Epoch [5/20] Train Acc: 0.9599 Val Acc: 0.9776
Epoch [6/20] Train Acc: 0.9633 Val Acc: 0.9784
Epoch [7/20] Train Acc: 0.9688 Val Acc: 0.9753
Epoch [8/20] Train Acc: 0.9715 Val Acc: 0.9776
Epoch [9/20] Train Acc: 0.9692 Val Acc: 0.9776
Epoch [10/20] Train Acc: 0.9711 Val Acc: 0.9792
Epoch [11/20] Train Acc: 0.9688 Val Acc: 0.9823
Epoch [12/20] Train Acc: 0.9706 Val Acc: 0.9784
Epoch [13/20] Train Acc: 0.9742 Val Acc: 0.9807
Epoch [14/20] Train Acc: 0.9704 Val Acc: 0.9815
Epoch [15/20] Train Acc: 0.9727 Val Acc: 0.9807
Epoch [16/20] Train Acc: 0.9738 Val Acc: 0.9800
Epoch [17/20] Train Acc: 0.9740 Val Acc: 0.9807
Epoch [18/20] Train Acc: 0.9793 Val Acc: 0.9769
Epoch [19/20] Train Acc: 0.9772 Val Acc: 0.9815
Epoch [20/20] Train Acc: 0.9729 Val Acc: 0.9823


In [45]:
torch.save(model.state_dict(), "mobilenet_balanced.pth")
print("Model saved successfully!")


Model saved successfully!


In [46]:
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

# Load test dataset
test_dataset = datasets.ImageFolder(f"{base_path}/test", transform=val_transform)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        _, predicted = torch.max(outputs, 1)

        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

accuracy = np.mean(np.array(all_preds) == np.array(all_labels))

print("Test Accuracy:", accuracy)
print("\nConfusion Matrix:")
print(confusion_matrix(all_labels, all_preds))
print("\nClassification Report:")
print(classification_report(all_labels, all_preds, target_names=test_dataset.classes))


Test Accuracy: 0.988078541374474

Confusion Matrix:
[[736   8]
 [  9 673]]

Classification Report:
              precision    recall  f1-score   support

          ai       0.99      0.99      0.99       744
        real       0.99      0.99      0.99       682

    accuracy                           0.99      1426
   macro avg       0.99      0.99      0.99      1426
weighted avg       0.99      0.99      0.99      1426



In [47]:
model = models.efficientnet_b0(pretrained=True)

# Freeze backbone
for param in model.features.parameters():
    param.requires_grad = False

# Replace classifier
in_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(in_features, 2)

model = model.to(device)

print("EfficientNet loaded successfully!")


/Users/indraneelpothuri/miniforge3/envs/aiml/lib/python3.11/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/Users/indraneelpothuri/miniforge3/envs/aiml/lib/python3.11/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=EfficientNet_B0_Weights.IMAGENET1K_V1`. You can also use `weights=EfficientNet_B0_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /Users/indraneelpothuri/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100.0%


EfficientNet loaded successfully!


In [48]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.classifier.parameters(), lr=1e-4)


In [49]:
num_epochs = 20

for epoch in range(num_epochs):
    model.train()
    train_correct = 0
    train_total = 0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        _, predicted = torch.max(outputs, 1)
        train_total += labels.size(0)
        train_correct += (predicted == labels).sum().item()

    train_acc = train_correct / train_total

    model.eval()
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            _, predicted = torch.max(outputs, 1)

            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()

    val_acc = val_correct / val_total

    print(f"Epoch [{epoch+1}/{num_epochs}] "
          f"Train Acc: {train_acc:.4f} "
          f"Val Acc: {val_acc:.4f}")


Epoch [1/20] Train Acc: 0.8143 Val Acc: 0.9190
Epoch [2/20] Train Acc: 0.9332 Val Acc: 0.9561
Epoch [3/20] Train Acc: 0.9528 Val Acc: 0.9614
Epoch [4/20] Train Acc: 0.9622 Val Acc: 0.9676
Epoch [5/20] Train Acc: 0.9647 Val Acc: 0.9715
Epoch [6/20] Train Acc: 0.9615 Val Acc: 0.9715
Epoch [7/20] Train Acc: 0.9692 Val Acc: 0.9753
Epoch [8/20] Train Acc: 0.9706 Val Acc: 0.9761
Epoch [9/20] Train Acc: 0.9706 Val Acc: 0.9800
Epoch [10/20] Train Acc: 0.9706 Val Acc: 0.9776
Epoch [11/20] Train Acc: 0.9742 Val Acc: 0.9807
Epoch [12/20] Train Acc: 0.9738 Val Acc: 0.9792
Epoch [13/20] Train Acc: 0.9738 Val Acc: 0.9784
Epoch [14/20] Train Acc: 0.9704 Val Acc: 0.9800
Epoch [15/20] Train Acc: 0.9781 Val Acc: 0.9800
Epoch [16/20] Train Acc: 0.9727 Val Acc: 0.9815
Epoch [17/20] Train Acc: 0.9747 Val Acc: 0.9800
Epoch [18/20] Train Acc: 0.9731 Val Acc: 0.9792
Epoch [19/20] Train Acc: 0.9774 Val Acc: 0.9838
Epoch [20/20] Train Acc: 0.9768 Val Acc: 0.9838


In [50]:
torch.save(model.state_dict(), "efficientnet_balanced.pth")
print("EfficientNet model saved!")


EfficientNet model saved!


In [51]:
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        _, predicted = torch.max(outputs, 1)

        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

accuracy = np.mean(np.array(all_preds) == np.array(all_labels))

print("Test Accuracy:", accuracy)

print("\nConfusion Matrix:")
print(confusion_matrix(all_labels, all_preds))

print("\nClassification Report:")
print(classification_report(all_labels, all_preds, target_names=test_dataset.classes))


Test Accuracy: 0.9726507713884993

Confusion Matrix:
[[717  27]
 [ 12 670]]

Classification Report:
              precision    recall  f1-score   support

          ai       0.98      0.96      0.97       744
        real       0.96      0.98      0.97       682

    accuracy                           0.97      1426
   macro avg       0.97      0.97      0.97      1426
weighted avg       0.97      0.97      0.97      1426



In [52]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

# Load pretrained ResNet50
model = models.resnet50(pretrained=True)

# Freeze backbone
for param in model.parameters():
    param.requires_grad = False

# Replace final layer
in_features = model.fc.in_features
model.fc = nn.Linear(in_features, 2)

model = model.to(device)

print("ResNet-50 loaded successfully!")


/Users/indraneelpothuri/miniforge3/envs/aiml/lib/python3.11/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/Users/indraneelpothuri/miniforge3/envs/aiml/lib/python3.11/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
2.2%

Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /Users/indraneelpothuri/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100.0%


ResNet-50 loaded successfully!


In [53]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.fc.parameters(), lr=1e-4)


In [54]:
num_epochs = 20

for epoch in range(num_epochs):
    model.train()
    train_correct = 0
    train_total = 0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        _, predicted = torch.max(outputs, 1)
        train_total += labels.size(0)
        train_correct += (predicted == labels).sum().item()

    train_acc = train_correct / train_total

    # Validation
    model.eval()
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            _, predicted = torch.max(outputs, 1)

            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()

    val_acc = val_correct / val_total

    print(f"Epoch [{epoch+1}/{num_epochs}] "
          f"Train Acc: {train_acc:.4f} "
          f"Val Acc: {val_acc:.4f}")


Epoch [1/20] Train Acc: 0.8532 Val Acc: 0.9561
Epoch [2/20] Train Acc: 0.9638 Val Acc: 0.9784
Epoch [3/20] Train Acc: 0.9715 Val Acc: 0.9838
Epoch [4/20] Train Acc: 0.9756 Val Acc: 0.9830
Epoch [5/20] Train Acc: 0.9765 Val Acc: 0.9854
Epoch [6/20] Train Acc: 0.9806 Val Acc: 0.9838
Epoch [7/20] Train Acc: 0.9786 Val Acc: 0.9854
Epoch [8/20] Train Acc: 0.9829 Val Acc: 0.9854
Epoch [9/20] Train Acc: 0.9793 Val Acc: 0.9869
Epoch [10/20] Train Acc: 0.9797 Val Acc: 0.9838
Epoch [11/20] Train Acc: 0.9820 Val Acc: 0.9884
Epoch [12/20] Train Acc: 0.9825 Val Acc: 0.9838
Epoch [13/20] Train Acc: 0.9813 Val Acc: 0.9830
Epoch [14/20] Train Acc: 0.9827 Val Acc: 0.9846
Epoch [15/20] Train Acc: 0.9831 Val Acc: 0.9869
Epoch [16/20] Train Acc: 0.9815 Val Acc: 0.9854
Epoch [17/20] Train Acc: 0.9838 Val Acc: 0.9869
Epoch [18/20] Train Acc: 0.9852 Val Acc: 0.9877
Epoch [19/20] Train Acc: 0.9854 Val Acc: 0.9900
Epoch [20/20] Train Acc: 0.9856 Val Acc: 0.9892


In [55]:
torch.save(model.state_dict(), "resnet50_balanced.pth")
print("ResNet-50 saved!")


ResNet-50 saved!


In [56]:
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        _, predicted = torch.max(outputs, 1)

        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

accuracy = np.mean(np.array(all_preds) == np.array(all_labels))

print("Test Accuracy:", accuracy)
print("\nConfusion Matrix:")
print(confusion_matrix(all_labels, all_preds))
print("\nClassification Report:")
print(classification_report(all_labels, all_preds, target_names=test_dataset.classes))


Test Accuracy: 0.988078541374474

Confusion Matrix:
[[735   9]
 [  8 674]]

Classification Report:
              precision    recall  f1-score   support

          ai       0.99      0.99      0.99       744
        real       0.99      0.99      0.99       682

    accuracy                           0.99      1426
   macro avg       0.99      0.99      0.99      1426
weighted avg       0.99      0.99      0.99      1426

